# Assignment 3: Gaussian Mixture Models and the EM Algorithm

This is the core notebook of the series. By the end you will have implemented a **Gaussian Mixture Model (GMM)** from scratch, including the full EM algorithm.

Topics covered (Lectures 3 & 4):
- GMM generative process
- Log-likelihood with the log-sum-exp trick
- The ELBO lower bound
- E-step: computing responsibilities
- M-step: updating parameters
- Convergence monitoring
- Visualisation and interpretation
- Failure modes: bad initialisation, singularities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms
from scipy.special import logsumexp

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 6)

---
## Part 1: The GMM Generative Process

A **Gaussian Mixture Model** with $K$ components defines:

$$p(\mathbf{x}) = \sum_{k=1}^K \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

where:
- $\pi_k \geq 0$, $\sum_k \pi_k = 1$ — **mixing coefficients** (prior probabilities of each component)
- $\boldsymbol{\mu}_k$ — component means
- $\boldsymbol{\Sigma}_k$ — component covariances

The **generative process** (ancestral sampling):
1. Sample a cluster index $z \sim \text{Categorical}(\boldsymbol{\pi})$
2. Sample the observation $\mathbf{x} \sim \mathcal{N}(\boldsymbol{\mu}_z, \boldsymbol{\Sigma}_z)$

The latent variable $z$ is not observed — that is why this is a **Latent Variable Model (LVM)**.

In [ ]:
def multivariate_gaussian_pdf(x, mu, Sigma):
    """
    Multivariate Gaussian PDF for a single data point.

    Args:
        x     : np.ndarray (D,)
        mu    : np.ndarray (D,)
        Sigma : np.ndarray (D, D)

    Returns:
        float, PDF value (NOT log)
    """
    D = len(mu)
    # TODO: implement the multivariate Gaussian PDF
    # Use np.linalg.det and np.linalg.inv
    # PDF = 1 / ((2pi)^(D/2) * |Sigma|^(1/2)) * exp(-0.5 * (x-mu)^T Sigma^{-1} (x-mu))
    raise NotImplementedError


def log_multivariate_gaussian_pdf(x, mu, Sigma):
    """
    Log of the multivariate Gaussian PDF — numerically more stable.

    Returns:
        float, log PDF value
    """
    D = len(mu)
    # TODO: compute the log PDF directly
    # log p = -D/2 * log(2pi) - 0.5 * log|Sigma| - 0.5 * (x-mu)^T Sigma^{-1} (x-mu)
    # Use np.linalg.slogdet for numerically stable log-determinant
    raise NotImplementedError

In [ ]:
def sample_gmm(pis, mus, Sigmas, n_samples):
    """
    Draw samples from a GMM using ancestral sampling.

    Args:
        pis      : np.ndarray (K,), mixing coefficients (must sum to 1)
        mus      : list of K np.ndarray (D,), component means
        Sigmas   : list of K np.ndarray (D,D), component covariances
        n_samples: int

    Returns:
        X : np.ndarray (n_samples, D)
        z : np.ndarray (n_samples,), integer cluster assignments
    """
    K = len(pis)
    D = len(mus[0])

    # TODO:
    # Step 1: sample cluster index z for each data point
    #         z ~ Categorical(pis)
    #         Hint: np.random.choice(K, size=n_samples, p=pis)
    z = None  # TODO

    # Step 2: for each data point, sample from the corresponding Gaussian
    #         x_n ~ N(mu_{z_n}, Sigma_{z_n})
    #         Hint: np.random.multivariate_normal(mu, Sigma)
    X = np.zeros((n_samples, D))
    for n in range(n_samples):
        # TODO: sample x_n given z_n
        pass

    return X, z


# Define a 3-component 2D GMM
pis_true   = np.array([0.3, 0.4, 0.3])
mus_true   = [np.array([-3, -2]), np.array([2, 2]), np.array([0, -3])]
Sigmas_true = [
    np.array([[1.0, 0.5], [0.5, 0.5]]),
    np.array([[0.5, -0.2], [-0.2, 1.0]]),
    np.array([[1.5, 0.0], [0.0, 0.3]]),
]

X, z_true = sample_gmm(pis_true, mus_true, Sigmas_true, n_samples=400)

# Plot samples
colors = ['tab:blue', 'tab:orange', 'tab:green']
plt.figure()
for k in range(3):
    mask = z_true == k
    plt.scatter(X[mask, 0], X[mask, 1], s=15, alpha=0.6, color=colors[k], label=f'Component {k+1}')
plt.title('Samples from a 3-component GMM (coloured by true cluster)')
plt.legend()
plt.axis('equal')
plt.show()

---
## Part 2: GMM Log-Likelihood

The log-likelihood of the GMM given data $\{\mathbf{x}_n\}_{n=1}^N$ is:

$$\ln p(\mathbf{X} | \boldsymbol{\pi}, \boldsymbol{\mu}, \boldsymbol{\Sigma}) = \sum_{n=1}^N \ln \left(\sum_{k=1}^K \pi_k \, \mathcal{N}(\mathbf{x}_n | \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)\right)$$

The log of a sum has **no closed form** for optimisation — this is why we need EM.

Numerically, we compute $\ln \sum_k \pi_k \mathcal{N}(\mathbf{x}_n | \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$ using the log-sum-exp trick:

$$\ln \sum_k e^{a_k} = \max_k(a_k) + \ln \sum_k e^{a_k - \max_k(a_k)}$$

In [ ]:
def gmm_log_likelihood(X, pis, mus, Sigmas):
    """
    Compute the total log-likelihood of the GMM.

    Args:
        X      : np.ndarray (N, D)
        pis    : np.ndarray (K,)
        mus    : list of K np.ndarray (D,)
        Sigmas : list of K np.ndarray (D,D)

    Returns:
        float, sum_{n=1}^N log p(x_n)
    """
    N, D = X.shape
    K = len(pis)
    total_ll = 0.0

    for n in range(N):
        # TODO:
        # 1. Compute log(pi_k) + log N(x_n | mu_k, Sigma_k) for each k
        #    (use log_multivariate_gaussian_pdf)
        # 2. Use log-sum-exp to compute log(sum_k pi_k N(x_n | ...))
        # 3. Add to total_ll
        pass

    return total_ll


# Test: log-likelihood on true parameters should be high
ll_true = gmm_log_likelihood(X, pis_true, mus_true, Sigmas_true)
print(f'Log-likelihood (true params): {ll_true:.2f}')

---
## Part 3: The EM Algorithm

The EM algorithm iterates between two steps to find the MLE parameters of the GMM:

**E-step** (Expectation): Compute the **responsibility** $r_{nk}$ — the posterior probability that component $k$ generated data point $x_n$:

$$r_{nk} = p(z_n = k \mid \mathbf{x}_n, \theta) = \frac{\pi_k \, \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_{j=1}^K \pi_j \, \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

**M-step** (Maximisation): Update parameters using the responsibilities as soft cluster assignments:

$$N_k = \sum_{n=1}^N r_{nk} \qquad \text{(effective number of points in cluster } k)$$

$$\pi_k^{\text{new}} = \frac{N_k}{N}$$

$$\boldsymbol{\mu}_k^{\text{new}} = \frac{1}{N_k} \sum_{n=1}^N r_{nk} \, \mathbf{x}_n$$

$$\boldsymbol{\Sigma}_k^{\text{new}} = \frac{1}{N_k} \sum_{n=1}^N r_{nk} (\mathbf{x}_n - \boldsymbol{\mu}_k^{\text{new}})(\mathbf{x}_n - \boldsymbol{\mu}_k^{\text{new}})^\top$$

In [ ]:
def e_step(X, pis, mus, Sigmas):
    """
    E-step: compute responsibilities r_{nk}.

    Args:
        X      : np.ndarray (N, D)
        pis    : np.ndarray (K,)
        mus    : list of K np.ndarray (D,)
        Sigmas : list of K np.ndarray (D,D)

    Returns:
        R : np.ndarray (N, K), responsibilities
            R[n, k] = r_{nk}  (each row sums to 1)
    """
    N, D = X.shape
    K = len(pis)
    log_R = np.zeros((N, K))

    # TODO:
    # For each n and k:
    #   log_R[n, k] = log(pi_k) + log_multivariate_gaussian_pdf(X[n], mus[k], Sigmas[k])
    # Then normalise each row:
    #   R[n, :] = softmax(log_R[n, :])  i.e., exp(log_R[n]) / sum(exp(log_R[n]))
    # Use the log-sum-exp trick for numerical stability in normalisation.

    # Hint: scipy.special.logsumexp(log_R, axis=1, keepdims=True) gives log(sum_k exp(log_R[n,k]))

    raise NotImplementedError


def m_step(X, R):
    """
    M-step: update GMM parameters given responsibilities.

    Args:
        X : np.ndarray (N, D)
        R : np.ndarray (N, K)

    Returns:
        pis_new    : np.ndarray (K,)
        mus_new    : list of K np.ndarray (D,)
        Sigmas_new : list of K np.ndarray (D,D)
    """
    N, D = X.shape
    K = R.shape[1]

    pis_new    = np.zeros(K)
    mus_new    = []
    Sigmas_new = []

    for k in range(K):
        # TODO:
        # 1. Compute N_k = sum_n r_{nk}
        N_k = None  # TODO

        # 2. Update pi_k = N_k / N
        pis_new[k] = None  # TODO

        # 3. Update mu_k = (1/N_k) * sum_n r_{nk} * x_n
        #    Vectorised: (R[:, k] @ X) / N_k
        mu_k = None  # TODO
        mus_new.append(mu_k)

        # 4. Update Sigma_k = (1/N_k) * sum_n r_{nk} * (x_n - mu_k)(x_n - mu_k)^T
        # Hint: diff = X - mu_k  has shape (N, D)
        #       Sigma_k = (diff.T * R[:, k]) @ diff / N_k
        diff = X - mu_k
        Sigma_k = None  # TODO
        Sigmas_new.append(Sigma_k)

    return pis_new, mus_new, Sigmas_new

### 3.1 Full EM loop

In [ ]:
def fit_gmm(X, K, n_iter=100, tol=1e-6, init='kmeans_like', random_state=None):
    """
    Fit a GMM to data X using the EM algorithm.

    Args:
        X            : np.ndarray (N, D)
        K            : int, number of components
        n_iter       : int, maximum EM iterations
        tol          : float, convergence threshold on log-likelihood change
        init         : str, initialisation strategy ('kmeans_like' or 'random')
        random_state : int or None

    Returns:
        pis    : np.ndarray (K,)
        mus    : list of K np.ndarray (D,)
        Sigmas : list of K np.ndarray (D,D)
        ll_history : list of float, log-likelihood per iteration
    """
    if random_state is not None:
        np.random.seed(random_state)

    N, D = X.shape

    # --- Initialisation ---
    if init == 'kmeans_like':
        # Pick K random data points as initial means
        idx = np.random.choice(N, K, replace=False)
        mus = [X[i].copy() for i in idx]
    else:
        # Random means from Gaussian
        mus = [np.random.randn(D) for _ in range(K)]

    pis    = np.ones(K) / K
    Sigmas = [np.eye(D) for _ in range(K)]

    ll_history = []

    for iteration in range(n_iter):
        # TODO: run one E-step and one M-step
        # 1. E-step: compute responsibilities
        # 2. M-step: update parameters
        # 3. Compute log-likelihood and append to ll_history
        # 4. Check convergence: if abs(ll_new - ll_old) < tol, break
        raise NotImplementedError

    return pis, mus, Sigmas, ll_history


# Fit the GMM
K = 3
pis_fit, mus_fit, Sigmas_fit, ll_history = fit_gmm(X, K=K, n_iter=100, random_state=7)

# Plot convergence
plt.plot(ll_history, 'b-o', markersize=3)
plt.xlabel('EM iteration')
plt.ylabel('Log-likelihood')
plt.title('EM convergence: log-likelihood over iterations')
plt.grid(True)
plt.show()

print(f'Converged in {len(ll_history)} iterations')
print(f'Final log-likelihood: {ll_history[-1]:.2f}')
print(f'Log-likelihood (true params): {gmm_log_likelihood(X, pis_true, mus_true, Sigmas_true):.2f}')

---
## Part 4: Visualisation of the Fitted GMM

In [ ]:
def plot_ellipse(ax, mu, Sigma, color, n_std=2.0, alpha=0.3, **kwargs):
    """Draw a confidence ellipse for a 2D Gaussian."""
    vals, vecs = np.linalg.eigh(Sigma)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    ellipse = Ellipse(xy=mu, width=width, height=height, angle=angle,
                      edgecolor=color, fc=color, alpha=alpha, **kwargs)
    ax.add_patch(ellipse)


def plot_gmm(X, pis, mus, Sigmas, title='GMM fit', z_true=None):
    fig, ax = plt.subplots()

    # Hard assignment for colouring
    R = e_step(X, pis, mus, Sigmas)
    z_pred = R.argmax(axis=1)

    colors_map = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
    for k in range(len(pis)):
        mask = z_pred == k
        ax.scatter(X[mask, 0], X[mask, 1], s=12, alpha=0.5,
                   color=colors_map[k], label=f'Cluster {k+1} (pi={pis[k]:.2f})')
        plot_ellipse(ax, mus[k], Sigmas[k], color=colors_map[k])
        ax.scatter(*mus[k], marker='x', s=100, color=colors_map[k], zorder=10)

    ax.set_title(title)
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    plt.show()


plot_gmm(X, pis_fit, mus_fit, Sigmas_fit, title='Fitted GMM (EM result)')

### 4.1 Compare fitted parameters to true parameters

In [ ]:
# Note: EM may permute cluster labels. The ordering below may not match.
print('True mixing coefficients:', pis_true)
print('Fitted mixing coefficients:', np.round(pis_fit, 3))
print()
for k in range(K):
    print(f'True mu_{k}: {mus_true[k]}')
    print(f'Fit  mu_{k}: {np.round(mus_fit[k], 2)}')
    print()

---
## Part 5: Failure Modes

### 5.1 Sensitivity to initialisation

EM is not guaranteed to find the global maximum — it finds a local optimum that depends on initialisation.

In [ ]:
# Run EM 10 times with different random seeds and compare final log-likelihoods
n_restarts = 10
results = []

for seed in range(n_restarts):
    pis_r, mus_r, Sigmas_r, ll_r = fit_gmm(X, K=3, n_iter=100, random_state=seed)
    results.append((ll_r[-1], pis_r, mus_r, Sigmas_r))

final_lls = [r[0] for r in results]
print('Final log-likelihoods across restarts:')
for i, ll in enumerate(final_lls):
    print(f'  Seed {i}: {ll:.2f}')

print(f'\nBest: {max(final_lls):.2f}, Worst: {min(final_lls):.2f}')
print(f'Range: {max(final_lls) - min(final_lls):.2f}')

# Plot the best and worst
best_idx  = np.argmax(final_lls)
worst_idx = np.argmin(final_lls)

_, pis_best, mus_best, Sigmas_best   = results[best_idx]
_, pis_worst, mus_worst, Sigmas_worst = results[worst_idx]

plot_gmm(X, pis_best,  mus_best,  Sigmas_best,  title=f'Best restart (seed={best_idx})')
plot_gmm(X, pis_worst, mus_worst, Sigmas_worst, title=f'Worst restart (seed={worst_idx})')

### 5.2 Wrong number of components

In [ ]:
# TODO: fit GMMs with K=2 and K=5 and plot the results
# Compare the log-likelihoods and discuss what happens with too few / too many components.
# Hint: why does a larger K always increase log-likelihood, even if the true K is smaller?

raise NotImplementedError

**Your discussion**: TODO

### 5.3 Singularities

A singularity occurs when a component collapses onto a single data point: $\boldsymbol{\Sigma}_k \to 0$, making the log-likelihood $\to +\infty$. This is a pathological solution.

A common fix is to add a small regularisation term to the diagonal of $\boldsymbol{\Sigma}$.

In [ ]:
def fit_gmm_regularised(X, K, n_iter=100, tol=1e-6, reg=1e-4, random_state=None):
    """
    GMM with regularisation to prevent singularities.

    Modify fit_gmm so that after each M-step:
        Sigma_k += reg * I

    This ensures Sigma_k remains invertible.

    TODO: copy your fit_gmm implementation here and add the regularisation.
    """
    raise NotImplementedError


# Test on a small dataset where singularity is likely
X_small = np.vstack([
    np.random.randn(10, 2),
    np.random.randn(10, 2) + 5
])

pis_r, mus_r, Sigmas_r, ll_r = fit_gmm_regularised(X_small, K=5, random_state=0)
plot_gmm(X_small, pis_r, mus_r, Sigmas_r, title='Regularised GMM (K=5 on small data)')

---
## Part 6: The ELBO and EM Connection

The ELBO for a GMM (with $q(z) = r_{nk}$) is:

$$\mathcal{L}(q, \theta) = \sum_{n=1}^N \sum_{k=1}^K r_{nk} \ln \frac{\pi_k \mathcal{N}(\mathbf{x}_n | \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{r_{nk}}$$

EM maximises this ELBO alternately over $q$ (E-step) and $\theta$ (M-step).

In [ ]:
def compute_elbo(X, R, pis, mus, Sigmas):
    """
    Compute the ELBO L(q, theta) for the GMM.

    Args:
        X      : np.ndarray (N, D)
        R      : np.ndarray (N, K), current responsibilities
        pis    : np.ndarray (K,)
        mus    : list of K np.ndarray (D,)
        Sigmas : list of K np.ndarray (D,D)

    Returns:
        float
    """
    # TODO: implement the ELBO formula above
    # Be careful about log(0): mask or clamp r_{nk}
    raise NotImplementedError


# Verify: ELBO <= log-likelihood at every step
# (run a few EM steps and compute both quantities)

N, D = X.shape
pis_init = np.ones(K) / K
mus_init = [X[i].copy() for i in np.random.choice(N, K, replace=False)]
Sigmas_init = [np.eye(D)] * K

elbos = []
lls   = []
pis_iter, mus_iter, Sigmas_iter = pis_init, mus_init, Sigmas_init

for _ in range(20):
    R_iter = e_step(X, pis_iter, mus_iter, Sigmas_iter)
    elbos.append(compute_elbo(X, R_iter, pis_iter, mus_iter, Sigmas_iter))
    lls.append(gmm_log_likelihood(X, pis_iter, mus_iter, Sigmas_iter))
    pis_iter, mus_iter, Sigmas_iter = m_step(X, R_iter)

plt.plot(lls,   'b-o', markersize=4, label='Log-likelihood')
plt.plot(elbos, 'r--s', markersize=4, label='ELBO')
plt.xlabel('EM iteration')
plt.title('Log-likelihood >= ELBO at every step')
plt.legend()
plt.show()

print('ELBO <= LL at every step:', all(e <= l + 1e-6 for e, l in zip(elbos, lls)))

---
## Part 7: GMM as a Density Estimator

Once fitted, we can use the GMM to **evaluate density** for new points and to **generate new samples**.

In [ ]:
def gmm_density(x_new, pis, mus, Sigmas):
    """
    Compute p(x_new) under the fitted GMM.

    Args:
        x_new  : np.ndarray (D,)
        pis    : np.ndarray (K,)
        mus    : list of K np.ndarray (D,)
        Sigmas : list of K np.ndarray (D,D)

    Returns:
        float
    """
    # TODO: sum over components:  sum_k pi_k * N(x_new | mu_k, Sigma_k)
    raise NotImplementedError


# Visualise density as a heatmap over the 2D plane
grid_range = np.linspace(-7, 7, 80)
GX, GY = np.meshgrid(grid_range, grid_range)
grid_points = np.column_stack([GX.ravel(), GY.ravel()])

density = np.array([gmm_density(p, pis_fit, mus_fit, Sigmas_fit) for p in grid_points])
density = density.reshape(GX.shape)

plt.figure(figsize=(8, 7))
plt.contourf(GX, GY, density, levels=30, cmap='Blues')
plt.colorbar(label='p(x)')
plt.scatter(X[:, 0], X[:, 1], s=5, color='red', alpha=0.3, label='Data')
plt.title('GMM density estimate')
plt.legend()
plt.axis('equal')
plt.show()

In [ ]:
# TODO: generate 200 new samples from the fitted GMM using your sample_gmm function
# and plot them alongside the original data.
# Do the generated samples look like the original data?

raise NotImplementedError

---
## Part 8: GMM on Real-ish Data — Iris Dataset

Let's apply the GMM to a well-known dataset and evaluate clustering quality.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X_iris = iris.data[:, :2]  # use first two features for 2D visualisation
y_iris = iris.target

scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# TODO:
# 1. Fit a GMM with K=3 on X_iris_scaled
# 2. Use plot_gmm to visualise the result
# 3. Compute the hard cluster assignments from responsibilities
# 4. Compare to the true labels y_iris — can you find a good permutation?
#    (EM doesn't know about class labels, so cluster indices may be permuted)
#    Hint: try all 6 permutations of (0,1,2) and pick the one with highest accuracy

raise NotImplementedError

---
## Summary

Congratulations — you have implemented a GMM from scratch! Here is what you built:

| Component | What it does |
|-----------|-------------|
| `sample_gmm` | Ancestral sampling from the GMM generative process |
| `gmm_log_likelihood` | Evaluates the data likelihood under the model |
| `e_step` | Computes soft cluster assignments (responsibilities) |
| `m_step` | Updates parameters from responsibilities |
| `fit_gmm` | Full EM loop with convergence monitoring |
| `compute_elbo` | Verifies the ELBO lower bound property |
| `gmm_density` | Evaluates the density for new points |

**Next**: Assignment 4 covers Probabilistic PCA and the Variational Autoencoder.